# Barycentric Lagrange Interpolation

Given distinct nodes $x_0, \ldots, x_n$ and values $y_0, \ldots, y_n$, the interpolating polynomial can be written in the (second) barycentric form
$$
p(x) = \frac{\displaystyle \sum_{j=0}^n \frac{y_j \lambda_j}{x - x_j}}{\displaystyle \sum_{j=0}^n \frac{\lambda_j}{x - x_j}},
\qquad
\lambda_j = \frac{1}{\prod_{k \neq j} (x_j - x_k)}.
$$
The formula is unchanged if all the weights are multiplied by the same constant.

In [ ]:
from math import comb

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def barycentric_eval(x, nodes, values, weights):
    diff = x[:, None] - nodes[None, :]
    i, j = np.nonzero(diff == 0)
    diff[i, j] = 1
    terms = weights / diff
    p = (terms @ values) / terms.sum(axis=1)
    p[i] = values[j]
    return p


def barycentric_interp(x, nodes, values):
    n = len(nodes) - 1
    weights = np.ones(n + 1)
    for j in range(n + 1):
        for k in range(n + 1):
            if k != j:
                weights[j] /= nodes[j] - nodes[k]
    return barycentric_eval(x, nodes, values, weights)

## Equispaced and Chebyshev nodes

For the **equispaced nodes** $x_j = a + jh$, $h = (b - a)/n$, we found (after cancelling a common factor) the weights
$$
\omega_j = (-1)^{n-j} \binom{n}{j}.
$$

For the **Chebyshev nodes** $x_j = \cos(j \pi / n)$ on $[-1, 1]$ (or mapped to $[a, b]$), the weights are, again up to a common factor,
$$
\omega_j = (-1)^j \delta_j, \qquad
\delta_j = \begin{cases} 1/2 & j = 0 \text{ or } j = n, \\ 1 & \text{otherwise.} \end{cases}
$$

In both cases, the weights are known explicitly, so they do not need to be computed.

In [ ]:
def equispaced_nodes(n, a=-1, b=1):
    return np.linspace(a, b, n + 1)


def chebyshev_nodes(n, a=-1, b=1):
    return (a + b) / 2 + (b - a) / 2 * np.cos(np.arange(n + 1) * np.pi / n)


def equispaced_interp(x, values, a=-1, b=1):
    n = len(values) - 1
    weights = np.array([(-1) ** (n - j) * comb(n, j) for j in range(n + 1)], dtype=float)
    return barycentric_eval(x, equispaced_nodes(n, a, b), values, weights)


def chebyshev_interp(x, values, a=-1, b=1):
    n = len(values) - 1
    weights = (-1.0) ** np.arange(n + 1)
    weights[0] /= 2
    weights[n] /= 2
    return barycentric_eval(x, chebyshev_nodes(n, a, b), values, weights)

## The Runge function

Let $f(x) = \dfrac{1}{1 + 25x^2}$ on $[-1, 1]$.
First, we interpolate at equispaced nodes. Try increasing $n$: the interpolant gets *worse* near the ends of the interval.

In [ ]:
def runge(x):
    return 1 / (1 + 25 * x**2)


n = 8

x = np.linspace(-1, 1, 1001)
nodes = equispaced_nodes(n)
p = equispaced_interp(x, runge(nodes))

plt.plot(x, runge(x), "k", linewidth=2, label="$f$")
plt.plot(x, p, label="$p$")
plt.plot(nodes, runge(nodes), "ko")
plt.xlabel("$x$")
plt.title(f"Equispaced nodes, $n = {n}$")
plt.legend()
plt.show()

With the same number of Chebyshev nodes, which cluster near the ends of the interval, the interpolant is much better, and it converges to $f$ as $n$ increases.

In [ ]:
nodes = chebyshev_nodes(n)
p = chebyshev_interp(x, runge(nodes))

plt.plot(x, runge(x), "k", linewidth=2, label="$f$")
plt.plot(x, p, label="$p$")
plt.plot(nodes, runge(nodes), "ko")
plt.xlabel("$x$")
plt.title(f"Chebyshev nodes, $n = {n}$")
plt.legend()
plt.show()